# Qualidade dos Dados: Olist

## Desafio de negócio

Identificar, **logo após a aprovação do pagamento**, quais pedidos possuem maior risco de serem
entregues **depois do prazo prometido**. A previsão deverá apoiar a priorização operacional, a
revisão do prazo e o acompanhamento preventivo dos pedidos.

## Objetivo deste notebook

Antes de qualquer modelo, precisamos saber **se podemos confiar nos dados**. Aqui vamos:

1. Entender a tabela principal, `olist_orders_dataset.csv`.
2. Investigar a base automaticamente com **FG Data Profiling**.
3. Criar um **contrato de qualidade** com **Pandera** avaliando as dimensões de qualidade.
4. A partir desse entendimento, olhar para as outras tabelas (`order_items` e `customers`).
5. Listar os **riscos de qualidade** e as decisões que eles geram para o modelo.

## Dimensões de qualidade avaliadas

| Dimensão | Pergunta |
|---|---|
| **Completude** | Os campos obrigatórios estão preenchidos? |
| **Unicidade** | Existe registro duplicado onde deveria haver um só? |
| **Validade** | Os valores respeitam o formato e o domínio esperado? |
| **Consistência** | As informações fazem sentido entre si (ex.: ordem das datas)? |
| **Pontualidade** | Os dados cobrem o período esperado e estão atualizados? |
| **Acurácia** | Os valores representam a realidade (ex.: preço positivo)? |

In [1]:
# Importando as bibliotecas
from pathlib import Path

import pandas as pd
import pandera.pandas as pa
import pandera.errors as pa_errors
from data_profiling import ProfileReport

pd.set_option("display.max_columns", 100)

# Caminhos do projeto (o notebook roda de dentro da pasta notebooks/)
ROOT = Path.cwd().parents[0]
print(ROOT)

DATA = ROOT / "data" / "raw"
REPORTS = ROOT / "reports"

d:\OneDrive\Área de Trabalho\curso_ml_cimatec\olist


## 1. Entendendo a tabela `orders`

Granularidade esperada: **1 linha = 1 pedido**. As colunas de data são lidas já como `datetime`.

In [2]:
# Colunas que representam datas do ciclo do pedido
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# carregando as orders
orders = pd.read_csv(
    DATA / "olist_orders_dataset.csv",
    parse_dates=DATE_COLUMNS,
)

print("Linhas x colunas:", orders.shape)
orders.head()

Linhas x colunas: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [3]:
# Tipos das colunas e quantidade de valores preenchidos
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [4]:
# Valores ausentes por coluna (quantidade e percentual)
missing = pd.DataFrame(
    {
        "ausentes": orders.isna().sum(),
        "percentual": (orders.isna().mean() * 100).round(2),
    }
)
missing

,ausentes,percentual
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


In [5]:
# Distribuição dos status dos pedidos
orders["order_status"].value_counts().to_frame("pedidos")

,pedidos
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


## 2. Investigação automática com FG Data Profiling

O **FG Data Profiling** (sucessor do *ydata-profiling*) gera um relatório com tipos, ausentes,
duplicados, distribuições e alertas de cada coluna.

Instalação no projeto: `uv add "fg-data-profiling[notebook]"`.

Usamos `minimal=True` para o relatório ficar leve. O HTML é salvo em `reports/`.

In [6]:
# Criar um profile do dataset
profile = ProfileReport(
    orders,
    title="Data Profiling - Olist Orders",
    minimal=True,
    explorative=False,
)

profile.to_file(REPORTS / "profile_olist_orders.html")

# Para ver dentro do notebook: profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:01<00:00,  4.92it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

## 3. Contrato de qualidade da tabela `orders` (Pandera)

O contrato descreve **como os dados deveriam ser**. Cada regra está ligada a uma dimensão de
qualidade. Usamos `lazy=True` para o Pandera coletar **todas** as violações antes de parar.

In [7]:
# ---------------------------------------------------------
# Domínio esperado do status do pedido
# ---------------------------------------------------------
STATUS_VALIDOS = [
    "created",
    "approved",
    "invoiced",
    "processing",
    "shipped",
    "delivered",
    "unavailable",
    "canceled",
]

# ---------------------------------------------------------
# Definição do contrato de qualidade da tabela orders
# ---------------------------------------------------------
orders_schema = pa.DataFrameSchema(
    {
        # -------------------------------------------------
        # UNICIDADE + COMPLETUDE + VALIDADE
        # -------------------------------------------------
        # 1 linha = 1 pedido, então order_id deve ser único.
        # Os ids da Olist são hashes de 32 caracteres.
        "order_id": pa.Column(
            str,
            pa.Check.str_length(32, 32),
            nullable=False,
            unique=True,
        ),

        # -------------------------------------------------
        # COMPLETUDE + VALIDADE
        # -------------------------------------------------
        # Todo pedido deve estar associado a um cliente.
        "customer_id": pa.Column(
            str,
            pa.Check.str_length(32, 32),
            nullable=False,
        ),

        # -------------------------------------------------
        # VALIDADE
        # -------------------------------------------------
        # O status deve pertencer ao processo da Olist.
        "order_status": pa.Column(
            str,
            pa.Check.isin(STATUS_VALIDOS),
            nullable=False,
        ),

        # -------------------------------------------------
        # COMPLETUDE
        # -------------------------------------------------
        # Precisamos saber quando o pedido foi realizado.
        "order_purchase_timestamp": pa.Column("datetime64[ns]", nullable=False),

        # Pode faltar em pedidos cancelados ou não aprovados.
        "order_approved_at": pa.Column("datetime64[ns]", nullable=True),

        # Nem todo pedido chegou à transportadora.
        "order_delivered_carrier_date": pa.Column("datetime64[ns]", nullable=True),

        # Nem todo pedido foi entregue.
        "order_delivered_customer_date": pa.Column("datetime64[ns]", nullable=True),

        # O prazo prometido é essencial: define se houve atraso.
        "order_estimated_delivery_date": pa.Column("datetime64[ns]", nullable=False),
    },
    checks=[
        # -------------------------------------------------
        # CONSISTÊNCIA entre colunas (ordem do ciclo do pedido)
        # Quando a data é nula, não consideramos violação.
        # -------------------------------------------------
        pa.Check(
            lambda df: df["order_approved_at"].isna()
            | (df["order_approved_at"] >= df["order_purchase_timestamp"]),
            name="aprovacao_depois_da_compra",
            error="order_approved_at não pode ocorrer antes de order_purchase_timestamp",
        ),
        pa.Check(
            lambda df: df["order_delivered_carrier_date"].isna()
            | df["order_approved_at"].isna()
            | (df["order_delivered_carrier_date"] >= df["order_approved_at"]),
            name="envio_depois_da_aprovacao",
            error="order_delivered_carrier_date não pode ocorrer antes da aprovação",
        ),
        pa.Check(
            lambda df: df["order_delivered_customer_date"].isna()
            | df["order_delivered_carrier_date"].isna()
            | (df["order_delivered_customer_date"] >= df["order_delivered_carrier_date"]),
            name="entrega_depois_do_envio",
            error="order_delivered_customer_date não pode ocorrer antes do envio",
        ),
        pa.Check(
            lambda df: df["order_estimated_delivery_date"] >= df["order_purchase_timestamp"],
            name="prazo_depois_da_compra",
            error="a entrega prometida não pode ser antes da compra",
        ),

        # -------------------------------------------------
        # COMPLETUDE condicional
        # Se o pedido está "delivered", ele precisa ter
        # data de aprovação e data de entrega ao cliente.
        # -------------------------------------------------
        pa.Check(
            lambda df: (df["order_status"] != "delivered")
            | df["order_delivered_customer_date"].notna(),
            name="entregue_tem_data_de_entrega",
            error="pedido delivered sem order_delivered_customer_date",
        ),
        pa.Check(
            lambda df: (df["order_status"] != "delivered")
            | df["order_approved_at"].notna(),
            name="entregue_tem_data_de_aprovacao",
            error="pedido delivered sem order_approved_at",
        ),
    ],
)

In [8]:
# ---------------------------------------------------------
# Validação
# ---------------------------------------------------------
# lazy=True: o Pandera encontra TODAS as violações antes de gerar o erro.
try:
    validated_orders = orders_schema.validate(orders, lazy=True)
    print("Dados válidos!")
    orders_failures = pd.DataFrame()

except pa_errors.SchemaErrors as err:
    print("Foram encontrados problemas de qualidade.")
    orders_failures = err.failure_cases

# Resumo: quantos PEDIDOS violam cada regra
# (uma regra entre colunas aparece uma vez por coluna envolvida,
#  por isso contamos os índices únicos)
if not orders_failures.empty:
    orders_failure_summary = (
        orders_failures.groupby("check")["index"]
        .nunique()
        .to_frame("pedidos_com_violacao")
        .sort_values("pedidos_com_violacao", ascending=False)
    )
    display(orders_failure_summary)

Foram encontrados problemas de qualidade.


,pedidos_com_violacao
check,
order_delivered_carrier_date não pode ocorrer antes da aprovação,1359
order_delivered_customer_date não pode ocorrer antes do envio,23
pedido delivered sem order_approved_at,14
pedido delivered sem order_delivered_customer_date,8


In [9]:
# Exemplos de pedidos que violam a ordem das datas
envio_antes_aprovacao = orders[
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
]

print("Envio antes da aprovação:", len(envio_antes_aprovacao))
envio_antes_aprovacao[
    ["order_id", "order_status", "order_approved_at", "order_delivered_carrier_date"]
].head()

Envio antes da aprovação: 1359


,order_id,order_status,order_approved_at,order_delivered_carrier_date
15,dcb36b511fcac050b97cd5c05de84dc3,delivered,2018-06-12 23:31:02,2018-06-11 14:54:00
64,688052146432ef8253587b930b01a06d,delivered,2018-04-24 18:25:22,2018-04-23 19:19:14
199,58d4c4747ee059eeeb865b349b41f53a,delivered,2018-07-26 23:31:53,2018-07-24 12:57:00
210,412fccb2b44a99b36714bca3fef8ad7b,delivered,2018-07-23 12:31:53,2018-07-23 12:24:00
415,56a4ac10a4a8f2ba7693523bb439eede,delivered,2018-07-27 23:31:09,2018-07-24 14:03:00


## 4. Pontualidade (cobertura temporal)

Verificamos o período coberto pela base e se o volume de pedidos é estável ao longo do tempo.
Meses com poucos pedidos geram taxas instáveis na análise.

In [10]:
print("Primeira compra:", orders["order_purchase_timestamp"].min())
print("Última compra:  ", orders["order_purchase_timestamp"].max())

pedidos_por_mes = (
    orders.set_index("order_purchase_timestamp")
    .resample("MS")
    .size()
    .to_frame("pedidos")
)

# Meses com menos de 100 pedidos (início e fim da base)
pedidos_por_mes.query("pedidos < 100")

Primeira compra: 2016-09-04 21:15:19
Última compra:   2018-10-17 17:30:18


,pedidos
order_purchase_timestamp,
2016-09-01,4
2016-11-01,0
2016-12-01,1
2018-09-01,16
2018-10-01,4


## 5. Olhando para as outras tabelas

O modelo vai usar também os **itens** (preço, frete, vendedores) e os **clientes** (estado).

In [11]:
items = pd.read_csv(DATA / "olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])

# zip_code_prefix é lido como texto: lido como número, perde o zero à esquerda (ex.: 01310).
customers = pd.read_csv(
    DATA / "olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": str},
)

print("Itens:", items.shape)
print("Clientes:", customers.shape)

Itens: (112650, 7)
Clientes: (99441, 5)


In [12]:
# ---------------------------------------------------------
# Contrato da tabela order_items (1 linha = 1 item do pedido)
# ---------------------------------------------------------
items_schema = pa.DataFrameSchema(
    {
        "order_id": pa.Column(str, pa.Check.str_length(32, 32), nullable=False),
        # VALIDADE: a numeração dos itens começa em 1
        "order_item_id": pa.Column(int, pa.Check.ge(1), nullable=False),
        "product_id": pa.Column(str, nullable=False),
        "seller_id": pa.Column(str, nullable=False),
        # ACURÁCIA: preço de produto vendido deve ser positivo
        "price": pa.Column(float, pa.Check.gt(0), nullable=False),
        # ACURÁCIA: frete não pode ser negativo (zero é possível: frete grátis)
        "freight_value": pa.Column(float, pa.Check.ge(0), nullable=False),
    },
    # UNICIDADE: um mesmo item não pode aparecer duas vezes no mesmo pedido
    unique=["order_id", "order_item_id"],
)

try:
    items_schema.validate(items, lazy=True)
    print("order_items: dados válidos!")
except pa_errors.SchemaErrors as err:
    print("order_items: problemas encontrados.")
    display(err.failure_cases.groupby(["column", "check"]).size().to_frame("violacoes"))

print("Itens com frete zero:", (items["freight_value"] == 0).sum())

order_items: dados válidos!
Itens com frete zero: 383


In [13]:
# ---------------------------------------------------------
# Contrato da tabela customers (1 linha = 1 customer_id)
# ---------------------------------------------------------
UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS", "MT", "PA",
    "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO",
]

customers_schema = pa.DataFrameSchema(
    {
        # UNICIDADE: cada pedido gera um customer_id próprio
        "customer_id": pa.Column(str, nullable=False, unique=True),
        "customer_unique_id": pa.Column(str, nullable=False),
        # VALIDADE: prefixo de CEP com 5 dígitos
        "customer_zip_code_prefix": pa.Column(
            str,
            pa.Check.str_matches(r"^\d{5}$"),
            nullable=False,
        ),
        "customer_city": pa.Column(str, nullable=False),
        # VALIDADE: estado deve ser uma UF brasileira
        "customer_state": pa.Column(str, pa.Check.isin(UFS), nullable=False),
    }
)

try:
    customers_schema.validate(customers, lazy=True)
    print("customers: dados válidos!")
except pa_errors.SchemaErrors as err:
    print("customers: problemas encontrados.")
    display(err.failure_cases.groupby(["column", "check"]).size().to_frame("violacoes"))

print(
    "CEPs que começam com 0 (perderiam o zero se lidos como número):",
    customers["customer_zip_code_prefix"].str.startswith("0").sum(),
)

customers: dados válidos!
CEPs que começam com 0 (perderiam o zero se lidos como número): 23995


## 6. Integridade entre as tabelas (consistência)

As tabelas serão unidas por `order_id` e `customer_id`. Precisamos saber se as chaves "batem".

In [14]:
integridade = pd.DataFrame(
    {
        "verificacao": [
            "pedidos sem nenhum item",
            "itens de pedidos inexistentes",
            "pedidos com cliente inexistente",
        ],
        "quantidade": [
            (~orders["order_id"].isin(items["order_id"])).sum(),
            (~items["order_id"].isin(orders["order_id"])).sum(),
            (~orders["customer_id"].isin(customers["customer_id"])).sum(),
        ],
    }
)
integridade

,verificacao,quantidade
0,pedidos sem nenhum item,775
1,itens de pedidos inexistentes,0
2,pedidos com cliente inexistente,0


In [15]:
# Que tipo de pedido fica sem itens?
orders.loc[~orders["order_id"].isin(items["order_id"]), "order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

## 7. Riscos de qualidade e decisões para o modelo

Resumo do que encontramos e do que isso significa para o problema de atraso.

In [16]:
def contar(mascara):
    return int(mascara.sum())


riscos = pd.DataFrame(
    [
        {
            "dimensao": "Completude",
            "verificacao": "order_approved_at ausente",
            "ocorrencias": contar(orders["order_approved_at"].isna()),
            "decisao": "A previsão acontece na aprovação: pedidos sem aprovação ficam fora.",
        },
        {
            "dimensao": "Completude",
            "verificacao": "delivered sem data de entrega",
            "ocorrencias": contar(
                orders["order_status"].eq("delivered")
                & orders["order_delivered_customer_date"].isna()
            ),
            "decisao": "Sem data real não dá para criar o alvo is_late: remover.",
        },
        {
            "dimensao": "Completude",
            "verificacao": "delivered sem data de aprovação",
            "ocorrencias": contar(
                orders["order_status"].eq("delivered") & orders["order_approved_at"].isna()
            ),
            "decisao": "Sem o instante da previsão (aprovação): remover da modelagem.",
        },
        {
            "dimensao": "Consistência",
            "verificacao": "envio antes da aprovação",
            "ocorrencias": contar(
                orders["order_delivered_carrier_date"] < orders["order_approved_at"]
            ),
            "decisao": "Data de envio é informação futura: não vira feature, só registramos.",
        },
        {
            "dimensao": "Consistência",
            "verificacao": "entrega antes do envio",
            "ocorrencias": contar(
                orders["order_delivered_customer_date"]
                < orders["order_delivered_carrier_date"]
            ),
            "decisao": "Poucos casos; mantidos, pois o alvo usa só entrega real x prometida.",
        },
        {
            "dimensao": "Consistência",
            "verificacao": "pedidos sem itens",
            "ocorrencias": contar(~orders["order_id"].isin(items["order_id"])),
            "decisao": "Quase todos não entregues: saem no filtro de pedidos entregues.",
        },
        {
            "dimensao": "Unicidade",
            "verificacao": "order_id duplicado",
            "ocorrencias": contar(orders["order_id"].duplicated()),
            "decisao": "Nenhuma ação: granularidade 1 pedido por linha confirmada.",
        },
        {
            "dimensao": "Validade",
            "verificacao": "CEP começando com 0",
            "ocorrencias": contar(customers["customer_zip_code_prefix"].str.startswith("0")),
            "decisao": "Ler o CEP como texto; usaremos o estado (customer_state) no modelo.",
        },
        {
            "dimensao": "Acurácia",
            "verificacao": "itens com frete zero",
            "ocorrencias": contar(items["freight_value"].eq(0)),
            "decisao": "Frete grátis é plausível: manter.",
        },
        {
            "dimensao": "Pontualidade",
            "verificacao": "meses com menos de 100 pedidos",
            "ocorrencias": len(pedidos_por_mes.query("pedidos < 100")),
            "decisao": "Cuidado ao analisar sazonalidade nas bordas da base (2016 e fim de 2018).",
        },
    ]
)

riscos

,dimensao,verificacao,ocorrencias,decisao
0,Completude,order_approved_at ausente,160,A previsão acontece na aprovação: pedidos sem ...
1,Completude,delivered sem data de entrega,8,Sem data real não dá para criar o alvo is_late...
2,Completude,delivered sem data de aprovação,14,Sem o instante da previsão (aprovação): remove...
3,Consistência,envio antes da aprovação,1359,Data de envio é informação futura: não vira fe...
4,Consistência,entrega antes do envio,23,"Poucos casos; mantidos, pois o alvo usa só ent..."
5,Consistência,pedidos sem itens,775,Quase todos não entregues: saem no filtro de p...
6,Unicidade,order_id duplicado,0,Nenhuma ação: granularidade 1 pedido por linha...
7,Validade,CEP começando com 0,23995,Ler o CEP como texto; usaremos o estado (custo...
8,Acurácia,itens com frete zero,383,Frete grátis é plausível: manter.
9,Pontualidade,meses com menos de 100 pedidos,5,Cuidado ao analisar sazonalidade nas bordas da...


## Conclusão

- A tabela `orders` tem **granularidade correta** (1 linha por pedido, `order_id` único) e
  status dentro do domínio esperado.
- Os ausentes nas datas são, em sua maioria, **esperados pelo processo** (pedidos cancelados ou
  ainda não entregues), mas existem pedidos `delivered` sem data de entrega ou de aprovação, que
  precisam sair da base de modelagem.
- Há **inconsistências na ordem das datas** (envio antes da aprovação e entrega antes do envio).
  Como essas datas são informações **futuras** em relação ao instante da previsão, elas não
  entram como features, mas reforçam a cautela com os registros logísticos.
- As chaves entre `orders`, `order_items` e `customers` são consistentes; os pedidos sem itens
  são quase todos não entregues.
- O CEP precisa ser lido como **texto**: lido como número, perde o zero à esquerda (CEPs de SP começam com 0).

**Decisão para o modelo:** usar apenas pedidos `delivered`, com `order_approved_at`,
`order_delivered_customer_date` e `order_estimated_delivery_date` preenchidos, e construir o alvo
`is_late` comparando entrega real e entrega prometida.